In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Set up the system: the harmonic oscillator, m = omega = 1.
M_MASS = 1.0
OMEGA  = 1.0
# Boundary data (nonzero and ASYMMETRIC on purpose: a symmetric
# choice q(T) = -q(0) would make the path odd about t = T/2, and a whole class
# of sign/indexing mistakes would cancel out and stay invisible).
QA, QB = 1.0, -0.5
T_END  = 2.0            # T = 2 < pi, so this is below the first conjugate point
H_of = lambda q, p: p**2/(2*M_MASS) + 0.5*M_MASS*OMEGA**2*q**2
print(f"m = {M_MASS},  omega = {OMEGA}")
print(f"q(0) = {QA},  q(T) = {QB},  T = {T_END}   (T/pi = {T_END/np.pi:.4f})")

Part 1 — Discretizing the phase-space action

Put q on the N+1 nodes kh and p on the N cell midpoints. That staggering is not decoration, q is naturally a cell quantity, so pairing p(k) with q(k+1)-q(k) keeps pdot times q second-order accurate without any interpolation.

Nothing anywhere below connects p to q. That is the whole experiment.

In [ ]:
# Set up the action function for the phase-space Lagrangian
def action_phase(q, p, h):
    """S[q,p] = sum_k [ p_k (q_{k+1}-q_k) - h H(qbar_k, p_k) ].

    q : (N+1,) values on nodes, INCLUDING both fixed endpoints
    p : (N,)   values on cell midpoints -- completely independent of q
    """
    qbar = 0.5*(q[1:] + q[:-1])
    return np.sum(p*(q[1:] - q[:-1]) - h*H_of(qbar, p))

def action_config(q, h):
    """The Week-4 object, for comparison: S[q] = sum_k h [ (m/2) qdot^2 - V ]."""
    qdot = (q[1:] - q[:-1])/h
    qbar = 0.5*(q[1:] + q[:-1])
    return np.sum(h*(0.5*M_MASS*qdot**2 - 0.5*M_MASS*OMEGA**2*qbar**2))

# Sanity check: feed in the exact continuum solution and confirm both actions
# agree.  They must -- on the true path (and only there) the two functionals
# take the same value.
c1      = (QB - QA*np.cos(OMEGA*T_END))/np.sin(OMEGA*T_END)
q_exact = lambda t: QA*np.cos(OMEGA*t) + c1*np.sin(OMEGA*t)
p_exact = lambda t: M_MASS*OMEGA*(-QA*np.sin(OMEGA*t) + c1*np.cos(OMEGA*t))

# closed form:  S* = (m w / 2) [ (qA^2+qB^2) cos(wT) - 2 qA qB ] / sin(wT)
S_EXACT = 0.5*M_MASS*OMEGA*((QA**2 + QB**2)*np.cos(OMEGA*T_END)
                            - 2*QA*QB)/np.sin(OMEGA*T_END)
N = 2000; h = T_END/N
t_nodes = np.linspace(0.0, T_END, N+1)
t_mid   = 0.5*(t_nodes[1:] + t_nodes[:-1])
Sp = action_phase(q_exact(t_nodes), p_exact(t_mid), h)
Sc = action_config(q_exact(t_nodes), h)
print(f"S_phase (exact path, N={N}) = {Sp:.12f}")
print(f"S_config(exact path, N={N}) = {Sc:.12f}")
print(f"S exact  (closed form)      = {S_EXACT:.12f}")
print(f"\nS_phase - S_config = {Sp-Sc:.2e}   <- must be ~0: on the TRUE path the")
print( "                                     two functionals agree identically.")
print(f"S_phase - S_exact  = {Sp-S_EXACT:.2e}   <- NOT zero: this is the O(h^2)")
print( "                                     quadrature error of the sum, and it")
print( "                                     shrinks by 4 each time N doubles.")

Solve for the stationary point, and watch the momentum appear

In [ ]:
def phase_system(N, T=T_END):
    """Assemble A x = -b for the stationary conditions.
    x = [ q_1 ... q_{N-1} , p_0 ... p_{N-1} ]   ->   2N-1 unknowns.
    The fixed endpoint values are NOT unknowns; wherever q_0 or q_N appears it
    moves to the right-hand side.  (Getting this wrong is the classic bug: if
    the endpoints leak into the unknown vector you silently solve a different
    boundary-value problem and everything still 'converges'.)
    """
    h = T/N; nq = N - 1; n = 2*N - 1
    A = np.zeros((n, n)); b = np.zeros(n)
    fixed = {0: QA, N: QB}
    def add_q(row, k, c):
        if k in fixed: b[row] += c*fixed[k]      # known -> right-hand side
        else:          A[row, k-1] += c          # unknown -> matrix
    for j in range(nq):                          # dS/dq_k = 0,  k = 1..N-1
        k = j + 1
        A[j, nq+k-1] += 1.0/h
        A[j, nq+k  ] -= 1.0/h
        add_q(j, k-1, -0.25*M_MASS*OMEGA**2)
        add_q(j, k  , -0.50*M_MASS*OMEGA**2)
        add_q(j, k+1, -0.25*M_MASS*OMEGA**2)
    for k in range(N):                           # dS/dp_k = 0,  k = 0..N-1
        r = nq + k
        add_q(r, k+1,  1.0/h)
        add_q(r, k  , -1.0/h)
        A[r, nq+k] -= 1.0/M_MASS
    return A, b, h, nq
def solve_phase(N, T=T_END):
    A, b, h, nq = phase_system(N, T)
    x = np.linalg.solve(A, -b)
    q = np.concatenate(([QA], x[:nq], [QB]))     # glue the fixed endpoints back on
    p = x[nq:]
    return q, p, h
print(f"{'N':>6} {'max|p - m dq/h|':>18} {'max|q - q_exact|':>18} {'ratio':>7} "
      f"{'S_phase':>16}")
prev = None
for N in [20, 40, 80, 160, 320, 640]:
    q, p, h = solve_phase(N)
    legendre = np.max(np.abs(p - M_MASS*(q[1:] - q[:-1])/h))
    err      = np.max(np.abs(q - q_exact(np.linspace(0.0, T_END, N+1))))
    ratio    = (prev/err) if prev else np.nan
    prev     = err
    print(f"{N:6d} {legendre:18.3e} {err:18.6e} {ratio:7.3f} "
          f"{action_phase(q, p, h):16.12f}")
print(f"\nS exact = {S_EXACT:.12f}")

In [ ]:
q80, p80, h80 = solve_phase(80)
tn = np.linspace(0.0, T_END, 81)
tm = 0.5*(tn[1:] + tn[:-1])

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))

ax[0].plot(tn, q_exact(tn), 'k-', lw=2.5, alpha=.35, label='exact $q(t)$')
ax[0].plot(tn, q80, 'o', ms=3.5, color='#ff8200', label='discrete $q_k$')
ax[0].plot([0, T_END], [QA, QB], 's', ms=9, mfc='none', mec='#c0392b', mew=2,
           label='fixed endpoints')
ax[0].set_xlabel('$t$'); ax[0].set_ylabel('$q$'); ax[0].legend(fontsize=8)
ax[0].set_title('position: solved for')

ax[1].plot(tm, p_exact(tm), 'k-', lw=2.5, alpha=.35, label='exact $p(t)$')
ax[1].plot(tm, p80, 'o', ms=3.5, color='#2e7d46', label='discrete $p_k$')
ax[1].plot(tm, M_MASS*(q80[1:]-q80[:-1])/h80, 'x', ms=5, color='#c0392b',
           label=r'$m\,\Delta q/h$  (never imposed)')
ax[1].set_xlabel('$t$'); ax[1].set_ylabel('$p$'); ax[1].legend(fontsize=8)
ax[1].set_title('momentum: NOT imposed, yet exact')

ax[2].plot(q80[:-1], p80, '-', color='#58595b', lw=1.4)
ax[2].plot(q80[0], p80[0], 'o', color='#2e7d46', ms=8, label='start')
ax[2].plot(q80[-2], p80[-1], 's', color='#c0392b', ms=8, label='end')
ax[2].set_xlabel('$q$'); ax[2].set_ylabel('$p$'); ax[2].legend(fontsize=8)
ax[2].set_title('the path in phase space')

plt.tight_layout(); plt.show()

Is the stationary point a minimum?

In [ ]:
def hessian_phase(N, T):
    """Hessian of the (h-scaled) phase-space action.  Size (2N-1).

    Scaling by a positive constant cannot change the SIGN of an eigenvalue,
    so the index we count below is a property of S itself.
    """
    h = T/N; nq = N - 1; n = 2*N - 1
    A = np.zeros((n, n))
    for j in range(nq):
        k = j + 1
        A[j, nq+k-1] += 1.0/h
        A[j, nq+k  ] -= 1.0/h
        for l, c in ((k-1, -0.25*M_MASS*OMEGA**2),
                     (k,   -0.50*M_MASS*OMEGA**2),
                     (k+1, -0.25*M_MASS*OMEGA**2)):
            if 1 <= l <= N-1:
                A[j, l-1] += c
    for k in range(N):
        r = nq + k
        if 1 <= k+1 <= N-1: A[r, k  ] += 1.0/h
        if 1 <= k   <= N-1: A[r, k-1] -= 1.0/h
        A[r, nq+k] -= 1.0/M_MASS
    return A

def hessian_config(N, T):
    """Hessian of the Week-4 configuration-space action.  Size (N-1)."""
    h = T/N; n = N - 1
    A = np.zeros((n, n))
    idx = lambda k: None if (k == 0 or k == N) else k - 1
    for k in range(N):                       # loop over cells
        a, b_ = idx(k), idx(k+1)
        for i, si in ((a, -1), (b_, 1)):     # kinetic: (m/h)(q_{k+1}-q_k)^2
            for j, sj in ((a, -1), (b_, 1)):
                if i is not None and j is not None:
                    A[i, j] += (M_MASS/h)*si*sj
        for i in (a, b_):                    # potential: -(h m w^2/4) at midpoint
            for j in (a, b_):
                if i is not None and j is not None:
                    A[i, j] += -(h*M_MASS*OMEGA**2/4.0)
    return A

n_neg = lambda A: int((np.linalg.eigvalsh(A) < -1e-9).sum())
N = 80
print(f"\n{'T':>7} {'T/pi':>7} {'dim':>6} {'phase: n_neg':>13} {'config: n_neg':>14}")
for T in [1.0, 2.0, 3.0, 4.0, 7.0, 10.0]:
    print(f"{T:7.2f} {T/np.pi:7.3f} {2*N-1:6d} "
          f"{n_neg(hessian_phase(N, T)):13d} {n_neg(hessian_config(N, T)):14d}")